In [ ]:
!pip install qiskit qiskit-aer matplotlib qiskit-ibm-runtime pylatexenc

In [1]:
#Import depenendencies
import numpy as np
from math import gcd, ceil, log2
from fractions import Fraction
from qiskit import QuantumCircuit, transpile
from qiskit_aer import Aer
from qiskit.circuit.library import UnitaryGate
from qiskit.visualization import plot_histogram
import matplotlib.pyplot as plt

In [2]:
#Computes the number of qubits required to represent integers modulo N.
def required_bits(N):
    return ceil(log2(N))

#Defining the unitary operator: |y⟩ → |(a·y mod N)⟩
#Modular_multiplication_unitary builds a reversible quantum operator that performs modular multiplication as a permutation of basis states, enabling Shor’s quantum period finding to work.
#The function constructs a permutation matrix that encodes modular multiplication by aaa modulo NNN as a reversible quantum operation.
def modular_multiplication_unitary(a, N, n_qubits):
    dim = 2 ** n_qubits        # Dimension of the Hilbert space
    U = np.zeros((dim, dim))   # Create an empty permutation matrix, which is automatically unitary.
    
    for y in range(dim):       #Populate the matrix
        if y < N:
            U[(a * y) % N, y] = 1  #∣y⟩↦∣(a⋅y) mod N⟩
        else:
            U[y, y] = 1            #∣y⟩↦∣y⟩ for y ≥ N
    return UnitaryGate(U, label=f"×{a} mod {N}")


#Defining the QFT inverse
def inverse_qft(qc, n):
    for i in range(n // 2):
        qc.swap(i, n - i - 1)
    for j in range(n):
        for m in range(j):
            qc.cp(-np.pi / (2 ** (j - m)), m, j)
        qc.h(j)

#Builds the entire quantum circuit for getting the period candidates of f(x) = a^x mod N function 
def build_shor_circuit(N, a):
    n_target = required_bits(N)
    n_count = n_target

    #Initializing the circuit
    qc = QuantumCircuit(n_count + n_target, n_count)
    qc.h(range(n_count))
    qc.x(n_count)

    #Applying the unitary operator: |y⟩ → |(a·y mod N)⟩
    for q in range(n_count):
        a_power = pow(a, 2**q, N)
        U = modular_multiplication_unitary(a_power, N, n_target)
        qc.append(
            U.control(),
            [q] + list(range(n_count, n_count + n_target))
        )
    #Applying the QFT inverse
    inverse_qft(qc, n_count)
    #Measuring the state
    qc.measure(range(n_count), range(n_count))
    return qc, n_count

#Get the factors of N from the period candidates of f(x) 
def classical_postprocess(counts, a, N, n_count):
    factors = set()

    for output in counts:
        decimal = int(output, 2)
        phase = decimal / (2 ** n_count)
        frac = Fraction(phase).limit_denominator(N)
        r = frac.denominator

        if r % 2 != 0:
            continue

        f1 = gcd(pow(a, r // 2) - 1, N)
        f2 = gcd(pow(a, r // 2) + 1, N)

        if 1 < f1 < N:
            factors.add(f1)
        if 1 < f2 < N:
            factors.add(f2)

    return list(factors)

In [3]:
#Defining the parameters, Number to factorize.
N = 21
a = 5
shots=1024

#Build the quantum circuit
qc, n_count = build_shor_circuit(N, a)
#Get the period candidates of f(x) = a^x mod N function 
backend = Aer.get_backend("qasm_simulator")
result = backend.run(transpile(qc, backend), shots=shots).result()
counts = result.get_counts()

#Show the Quantum Circuit of Shor's Algorithm
print("Quantum Circuit of Shor's Algorithm")
print( qc.draw("text") )

# Show the histogram
plot_histogram(counts)
plt.show()

factors = classical_postprocess(counts, a, N, n_count)

if factors:
    print("Factors found:", factors)
else:
    print("No non-trivial factors found.")

Quantum Circuit of Shor's Algorithm
     ┌───┐                                                         »
q_0: ┤ H ├──────■──────────────────────────────────────────────────»
     ├───┤      │                                                  »
q_1: ┤ H ├──────┼─────────────■────────────────────────────────────»
     ├───┤      │             │                                    »
q_2: ┤ H ├──────┼─────────────┼──────────────■─────────────────────»
     ├───┤      │             │              │                     »
q_3: ┤ H ├──────┼─────────────┼──────────────┼─────────────■───────»
     ├───┤      │             │              │             │       »
q_4: ┤ H ├──────┼─────────────┼──────────────┼─────────────┼───────»
     ├───┤┌─────┴──────┐┌─────┴──────┐┌──────┴──────┐┌─────┴──────┐»
q_5: ┤ X ├┤0           ├┤0           ├┤0            ├┤0           ├»
     └───┘│            ││            ││             ││            │»
q_6: ─────┤1           ├┤1           ├┤1            ├┤1           ├

Executing the simulation using IBM Quantum hardware

In [6]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime import SamplerV2


QiskitRuntimeService.save_account(token="YOUR KEY", overwrite=True)

service = QiskitRuntimeService()
print(service.backends())
#select a free computer
backend = service.least_busy(simulator=False, operational=True)

#Defining the parameters, Number to factorize.
N = 6
a = 5
shots=1024
#Build the quantum circuit
qc, n_count = build_shor_circuit(N, a)

#Compile the cicuit for run it into the IBM cimputer
pm = generate_preset_pass_manager(target=backend.target, optimization_level=1)
compiled_circuit = pm.run(qc)
print(compiled_circuit.draw("text"))

#Run Execution
sampler = SamplerV2(mode=backend)
job=sampler.run([compiled_circuit], shots=1024)
result = job.result()
counts = result[0].data.c.get_counts()

# Show the histogram
plot_histogram(counts)
plt.show()

factors = classical_postprocess(counts, a, N, n_count)

if factors:
    print("Factors found:", factors)
else:
    print("No non-trivial factors found.")



qiskit_runtime_service.__init__:WARNING:2026-04-24 07:47:29,085: Instance was not set at service instantiation. Free and trial plan instances will be prioritized. Based on the following filters: (tags: None, region: us-east, eu-de), and available plans: (open), the available account instances are: open-instance. If you need a specific instance set it explicitly either by using a saved account with a saved default instance or passing it in directly to QiskitRuntimeService().
qiskit_runtime_service.backends:WARNING:2026-04-24 07:47:29,086: Loading instance: open-instance, plan: open


[<IBMBackend('ibm_fez')>, <IBMBackend('ibm_kingston')>, <IBMBackend('ibm_marrakesh')>]


qiskit_runtime_service.backends:WARNING:2026-04-24 07:47:30,872: Loading instance: open-instance, plan: open
qiskit_runtime_service.backends:WARNING:2026-04-24 07:47:31,922: Using instance: open-instance, plan: open


global phase: 5π/16
           ┌─────────────┐┌────┐┌──────────────┐┌────┐ ┌────────────┐    »
 q_0 -> 97 ┤ Rz(-3.1217) ├┤ √X ├┤ Rz(-0.99309) ├┤ √X ├─┤ Rz(2.0575) ├──■─»
           └──┬────────┬─┘├────┤└─┬─────────┬──┘└────┘ └────────────┘  │ »
q_2 -> 105 ───┤ Rz(-π) ├──┤ √X ├──┤ Rz(π/2) ├──────────────────────────┼─»
            ┌─┴────────┴─┐├────┤  ├─────────┤                          │ »
q_5 -> 106 ─┤ Rz(0.8622) ├┤ √X ├──┤ Rz(π/2) ├──────────────────────────┼─»
           ┌┴────────────┤├────┤ ┌┴─────────┴─┐ ┌────┐┌──────────────┐ │ »
q_4 -> 107 ┤ Rz(-2.2553) ├┤ √X ├─┤ Rz(-1.403) ├─┤ √X ├┤ Rz(-0.51138) ├─■─»
           └─┬─────────┬─┘├────┤ └┬──────────┬┘ └────┘└──────────────┘   »
q_3 -> 108 ──┤ Rz(π/2) ├──┤ √X ├──┤ Rz(-π/2) ├───────────────────────────»
             └┬────────┤  ├────┤  ├─────────┬┘                           »
q_1 -> 117 ───┤ Rz(-π) ├──┤ √X ├──┤ Rz(π/2) ├────────────────────────────»
              └────────┘  └────┘  └─────────┘                            »
     